# 03 · Inference with the packaged model
The champion is saved in three interchangeable formats:
* `models/heart_model.joblib` – scikit-learn Pipeline (preprocessing + model), used by the API
* `models/mlflow_model/` – MLflow model (pyfunc flavour, signature + input example)
* `models/heart_model.onnx` – ONNX graph including preprocessing, for framework-free runtimes

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
pd.set_option("display.width", 140)
import json
from heart.config import FEATURES
from heart.predict import HeartModel
patients = [json.loads((ROOT / "samples" / f).read_text()) for f in ["patient.json", "patient_high_risk.json"]]
pd.DataFrame(patients)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0,6
1,67,1,4,160,286,0,2,108,1,1.5,2,3,3


## 1. joblib pipeline through the `HeartModel` wrapper (same code path as the API)

In [2]:
model = HeartModel()
print("version:", model.version, "| algorithm:", model.metadata["algorithm"])
pd.DataFrame(model.predict_records(patients))

version: 2 | algorithm: logistic_regression


,prediction,label,probability_disease,confidence
0,0,no heart disease,0.3526,0.6474
1,1,heart disease,0.9928,0.9928


## 2. MLflow pyfunc model

In [3]:
import mlflow.pyfunc
pyfunc = mlflow.pyfunc.load_model(str(ROOT / "models/mlflow_model"))
X = pd.DataFrame(patients)[FEATURES].astype(float)
pyfunc.predict(X)

array([0, 1])

## 3. ONNX runtime (no scikit-learn needed) – parity check

In [4]:
import onnxruntime as rt
sess = rt.InferenceSession(str(ROOT / "models/heart_model.onnx"), providers=["CPUExecutionProvider"])
test = pd.read_csv(ROOT / "data/processed/test.csv")[FEATURES]
onnx_p = sess.run(None, {c: test[[c]].to_numpy(np.float32) for c in FEATURES})[1][:, 1]
skl_p = model.pipeline.predict_proba(test.astype(float))[:, 1]
print(f"max |onnx - sklearn| over {len(test)} test rows: {np.abs(onnx_p - skl_p).max():.2e}")

max |onnx - sklearn| over 61 test rows: 9.15e-08


## 4. The REST API (in-process, identical to the container)

In [5]:
from fastapi.testclient import TestClient
from app.main import app
with TestClient(app) as client:
    r = client.post("/predict", json=patients[1])
    print(r.status_code, r.json())
    bad = client.post("/predict", json={**patients[1], "cp": 9})
    print(bad.status_code, bad.json()["detail"][0]["msg"])

{"ts": "2026-10-17T17:27:28", "level": "INFO", "logger": "heart-disease-api", "msg": "model loaded", "model_path": "C:\\Users\\MSUSERSL123\\Downloads\\LLM\\heart-disease-mlops\\heart-disease-mlops\\models\\heart_model.joblib", "model_version": "2", "algorithm": "logistic_regression"}


{"ts": "2026-10-17T17:27:28", "level": "INFO", "logger": "heart-disease-api", "msg": "prediction", "inputs": {"age": 67.0, "sex": 1, "cp": 4, "trestbps": 160.0, "chol": 286.0, "fbs": 0, "restecg": 2, "thalach": 108.0, "exang": 1, "oldpeak": 1.5, "slope": 2, "ca": 3, "thal": 3}, "prediction": 1, "label": "heart disease", "probability_disease": 0.9928, "confidence": 0.9928, "model_version": "2"}


{"ts": "2026-10-17T17:27:28", "level": "INFO", "logger": "heart-disease-api", "msg": "request", "request_id": "7d65a3900191", "method": "POST", "path": "/predict", "status": 200, "latency_ms": 14.05, "client": "testclient"}


200 {'prediction': 1, 'label': 'heart disease', 'probability_disease': 0.9928, 'confidence': 0.9928, 'model_version': '2'}
{"ts": "2026-10-17T17:27:28", "level": "INFO", "logger": "heart-disease-api", "msg": "request", "request_id": "a1324d4993b4", "method": "POST", "path": "/predict", "status": 422, "latency_ms": 0.57, "client": "testclient"}


422 Input should be 1, 2, 3 or 4


Equivalent `curl` against the container or the Kubernetes service:
```bash
curl -X POST http://localhost:8000/predict -H 'content-type: application/json' -d @samples/patient.json
```